Extract miRNA seed; 1-based positions 2-7 of miRNA

In [ ]:
import pandas as pd
from miRBench.dataset import get_dataset_df 


df1 = get_dataset_df('AGO2_CLASH_Hejret2023', split="test")[['gene', 'noncodingRNA', 'label']]
df2 = get_dataset_df('AGO2_eCLIP_Klimentova2022', split="test")[['gene', 'noncodingRNA', 'label']]
datasets = {"AGO2_CLASH_Hejret2023": df1,
    "AGO2_eCLIP_Klimentova2022": df2}


def seed6mer(noncodingRNA):
    return noncodingRNA[1:7]

for name, df in datasets.items():
    df["seed6mer"] = df["noncodingRNA"].apply(seed6mer)
    print(f"\n--- {name} ---")
    display (df['seed6mer'].head())

Reverse complement (rv) the extracted seed sequence

In [ ]:
Comp =  str.maketrans({"A": "T", "T": "A", "C": "G", "G": "C"})

for name, df in datasets.items():
    df["seed6mer_c"] = df['seed6mer'].str.translate(Comp)
    df['seed6mer_rv'] = df['seed6mer_c'].str[::-1]

    print(name)
    display(df[['seed6mer','seed6mer_c','seed6mer_rv']].head())

In [ ]:
from Bio import Align
aligner = Align.PairwiseAligner()
aligner.mode = "global"

print(aligner)
aligner.open_left_deletion_score = 0.000
aligner.extend_left_deletion_score = 0.000
aligner.open_right_deletion_score = 0.000
aligner.extend_right_deletion_score = 0.000
print(aligner)

def get_score(row):
    return aligner.score(row["gene"], row["seed6mer_rv"])

for name, df in datasets.items():
    print(f'-------{name}------')
    df["score"] = df.apply(get_score, axis=1)
    display(df[["gene", "noncodingRNA", "score"]].head())